In [1]:

!git clone https://github.com/AndrewJoe0/CAP_3764_2026_FALL_TEAM_3.git


Cloning into 'CAP_3764_2026_FALL_TEAM_3'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 62 (delta 18), reused 31 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 435.43 KiB | 2.39 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [2]:

%cd /content/CAP_3764_2026_FALL_TEAM_3


/content/CAP_3764_2026_FALL_TEAM_3


In [3]:

import pandas as pd

from src.preprocessing import clean_data, time_split

df = clean_data(
    pd.read_csv("data/us_average_prices_monthly.csv")
)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())


Total rows: 8869
Columns: ['date', 'year', 'month', 'item', 'unit', 'category', 'price', 'series_id']


,date,year,month,item,unit,category,price,series_id
0,2015-01-01,2015,1,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.211,APU0000FD2101
1,2015-02-01,2015,2,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.221,APU0000FD2101
2,2015-03-01,2015,3,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.176,APU0000FD2101
3,2015-04-01,2015,4,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",2.971,APU0000FD2101
4,2015-05-01,2015,5,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.045,APU0000FD2101


In [4]:

train, test = time_split(df, "2024-01-01")

print("Training rows:", len(train))
print("Testing rows:", len(test))


Training rows: 6971
Testing rows: 1898


In [5]:

features = ["item", "year", "month"]

X_train = train[features]
X_test = test[features]

y_train = train["price"]
y_test = test["price"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


Training features: (6971, 3)
Testing features: (1898, 3)


In [6]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            ["item"]
        ),
        (
            "numeric",
            StandardScaler(),
            ["year", "month"]
        )
    ]
)

print("Preprocessing ready!")


Preprocessing ready!


In [7]:

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from src.evaluation import evaluate_regression

linear_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_results = evaluate_regression(
    y_test,
    linear_predictions
)

print("Linear Regression Results:")
print(linear_results)


Linear Regression Results:
{'RMSE': np.float64(0.9590249166037595), 'MAE': 0.6581595241679534, 'R2': 0.8988347467895256}


In [12]:

import numpy as np
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)

ridge_results = evaluate_regression(
    y_test,
    ridge_predictions
)

print("Ridge Regression Results:")
print(ridge_results)


Ridge Regression Results:
{'RMSE': np.float64(0.9770105903572551), 'MAE': 0.6769888500675276, 'R2': 0.8950046334940609}


In [9]:

from sklearn.linear_model import Lasso

lasso_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(alpha=0.1, max_iter=20000))
])

lasso_model.fit(X_train, y_train)

lasso_predictions = lasso_model.predict(X_test)

lasso_results = evaluate_regression(
    y_test,
    lasso_predictions
)

print("Lasso Regression Results:")
print(lasso_results)


Lasso Regression Results:
{'RMSE': np.float64(2.925329989524834), 'MAE': 2.2843207111045807, 'R2': 0.058714609543415675}


In [10]:

results_df = pd.DataFrame([
    {"Model": "Linear Regression", **linear_results},
    {"Model": "Ridge Regression", **ridge_results},
    {"Model": "Lasso Regression", **lasso_results}
])

display(results_df.round(4))


,Model,RMSE,MAE,R2
0,Linear Regression,0.9590,0.6582,0.8988
1,Ridge Regression,0.9770,0.6770,0.8950
2,Lasso Regression,2.9253,2.2843,0.0587


In [13]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

# Sort training data by date.
train_sorted = train.sort_values("date").copy()

X_train_sorted = train_sorted[features]
y_train_sorted = train_sorted["price"]

# Create time-based validation folds.
dates = train_sorted["date"].drop_duplicates().sort_values()

folds = []

for i in range(1, 4):
    split_point = len(dates) * i // 4
    next_point = len(dates) * (i + 1) // 4

    train_dates = dates.iloc[:split_point]
    val_dates = dates.iloc[split_point:next_point]

    train_idx = np.where(
        train_sorted["date"].isin(train_dates)
    )[0]

    val_idx = np.where(
        train_sorted["date"].isin(val_dates)
    )[0]

    folds.append((train_idx, val_idx))

ridge_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Ridge())
])

ridge_search = GridSearchCV(
    ridge_pipeline,
    {"model__alpha": [0.01, 0.1, 1.0, 10.0]},
    cv=folds,
    scoring="neg_root_mean_squared_error"
)

ridge_search.fit(X_train_sorted, y_train_sorted)

ridge_tuned_predictions = ridge_search.predict(X_test)

ridge_tuned_results = evaluate_regression(
    y_test,
    ridge_tuned_predictions
)

print("Best Ridge Alpha:", ridge_search.best_params_)
print("Tuned Ridge Results:", ridge_tuned_results)


Best Ridge Alpha: {'model__alpha': 0.01}
Tuned Ridge Results: {'RMSE': np.float64(0.9591881090123902), 'MAE': 0.6583660071058527, 'R2': 0.8988003143035372}


In [14]:

lasso_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(max_iter=20000))
])

lasso_search = GridSearchCV(
    lasso_pipeline,
    {"model__alpha": [0.0001, 0.001, 0.01, 0.1]},
    cv=folds,
    scoring="neg_root_mean_squared_error"
)

lasso_search.fit(X_train_sorted, y_train_sorted)

lasso_tuned_predictions = lasso_search.predict(X_test)

lasso_tuned_results = evaluate_regression(
    y_test,
    lasso_tuned_predictions
)

print("Best Lasso Alpha:", lasso_search.best_params_)
print("Tuned Lasso Results:", lasso_tuned_results)


Best Lasso Alpha: {'model__alpha': 0.0001}
Tuned Lasso Results: {'RMSE': np.float64(0.9638028733758809), 'MAE': 0.6640133243560891, 'R2': 0.8978242051865115}


In [15]:

final_results = pd.DataFrame([
    {"Model": "Linear Regression", **linear_results},
    {"Model": "Ridge (Original)", **ridge_results},
    {"Model": "Ridge (Tuned)", **ridge_tuned_results},
    {"Model": "Lasso (Original)", **lasso_results},
    {"Model": "Lasso (Tuned)", **lasso_tuned_results}
])

final_results = final_results.sort_values("RMSE")

display(final_results.round(4))


,Model,RMSE,MAE,R2
0,Linear Regression,0.9590,0.6582,0.8988
2,Ridge (Tuned),0.9592,0.6584,0.8988
4,Lasso (Tuned),0.9638,0.6640,0.8978
1,Ridge (Original),0.9770,0.6770,0.8950
3,Lasso (Original),2.9253,2.2843,0.0587


## Model Results and Conclusion

We trained and evaluated three regression models: Linear Regression, Ridge Regression, and Lasso Regression. The models used item type, year, and month as features to predict monthly grocery and gasoline prices.

We used a time-based split, with observations before January 2024 used for training and observations from January 2024 onward used for testing. Ridge and Lasso were tuned using time-based cross-validation on the training data.

**Linear Regression achieved the best overall performance**, with an RMSE of 0.9590, MAE of 0.6582, and R² of 0.8988.

Tuned Ridge Regression (alpha = 0.01) achieved an RMSE of 0.9592, while tuned Lasso Regression (alpha = 0.0001) achieved an RMSE of 0.9638.

These results suggest that Linear Regression provides the best performance among the tested models, although the tuned Ridge and Lasso models performed similarly. The results also demonstrate that adjusting regularization strength significantly improved Lasso's performance.